# 15 — SQLite, SQL, and Transactions

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "intermediate", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
from lesson import Exercises, ___, run

check = Exercises('intermediate/15', {1: '05568e7ffda76316', 2: '514cea2007eb991e', 3: '0f2aa04b0b818804', 4: 'e1059be804de509b', 5: '18e5958a407b78b7', 6: 'fa414ce61747f6f1', 7: '4944b8a215b93da0', 8: 'a8291e7a9d48b915', 9: 'bc0ea1ba5d261224', 10: '27160bddd7f362e6'}, practice='intermediate_04_sql_http_cli', independent='count_by_status')

## 1. Connect and create

- `sqlite3.connect(":memory:")` makes a throwaway database; pass a filename to keep it.
- `conn.execute(sql)` runs one statement; `conn.executescript(sql)` runs several.
- Constraints guard the data: `PRIMARY KEY`, `NOT NULL`, `DEFAULT 0`, `CHECK (balance >= 0)`.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.executescript("""
CREATE TABLE tasks (id INTEGER PRIMARY KEY, title TEXT NOT NULL, done INTEGER NOT NULL DEFAULT 0);
CREATE TABLE accounts (name TEXT PRIMARY KEY, balance REAL NOT NULL CHECK (balance >= 0));
""")
print([r[0] for r in conn.execute("SELECT name FROM sqlite_master ORDER BY name")])

**Exercise 1.** Which method runs a script with several statements? Write its name.

In [ ]:
method = ___
check(1, method)

**Exercise 2.** Insert a task without `done`. What does the `DEFAULT` give it?

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE tasks (id INTEGER PRIMARY KEY, title TEXT, done INTEGER NOT NULL DEFAULT 0)")
conn.execute("INSERT INTO tasks (title) VALUES ('write')")
predicted = ___   # the done value of that row
check(2, predicted)

**Exercise 3.** A `CHECK (balance >= 0)` rejects a negative balance. Which exception class? Write its name.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE a (balance REAL CHECK (balance >= 0))")
try:
    conn.execute("INSERT INTO a VALUES (-5)")
except sqlite3.Error as error:
    name = type(error).__name__
predicted = ___
check(3, predicted)

## 2. CRUD with placeholders

- Always pass values with `?` placeholders: `conn.execute("INSERT INTO tasks (title) VALUES (?)", (title,))`. Never build SQL with f-strings (SQL injection).
- `cursor.lastrowid` is the new row's id; `cursor.rowcount` is how many rows an UPDATE/DELETE changed.
- `fetchall()` returns tuples; `ORDER BY id` makes the order reliable. `LIKE ?` with `f"%{text}%"` searches.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE tasks (id INTEGER PRIMARY KEY, title TEXT, done INTEGER DEFAULT 0)")
for title in ["write tests", "ship", "Write docs"]:
    cur = conn.execute("INSERT INTO tasks (title) VALUES (?)", (title,))
print("last id:", cur.lastrowid)
print(conn.execute("UPDATE tasks SET done = 1 WHERE id = ?", (2,)).rowcount)
print(conn.execute("SELECT title FROM tasks WHERE title LIKE ? ORDER BY id", ("%write%",)).fetchall())

**Exercise 4.** Insert two tasks. What is `lastrowid` after the second?

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE tasks (id INTEGER PRIMARY KEY, title TEXT)")
conn.execute("INSERT INTO tasks (title) VALUES (?)", ("a",))
cur = conn.execute("INSERT INTO tasks (title) VALUES (?)", ("b",))
predicted = ___
check(4, predicted)

**Exercise 5.** Delete a task that doesn't exist. What is `rowcount`?

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE tasks (id INTEGER PRIMARY KEY, title TEXT)")
count = conn.execute("DELETE FROM tasks WHERE id = ?", (99,)).rowcount
predicted = ___
check(5, predicted)

**Exercise 6.** Return just the titles as a list of strings.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE tasks (id INTEGER PRIMARY KEY, title TEXT)")
conn.executemany("INSERT INTO tasks (title) VALUES (?)", [("a",), ("b",)])
rows = conn.execute("SELECT title FROM tasks ORDER BY id").fetchall()
titles = [___ for row in rows]
check(6, titles)

**Exercise 7.** Search titles containing `doc` with `LIKE` (SQLite's LIKE ignores ASCII case).

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE tasks (id INTEGER PRIMARY KEY, title TEXT)")
conn.executemany("INSERT INTO tasks (title) VALUES (?)", [("Docs",), ("code",), ("update doc",)])
pattern = "%" + ___ + "%"
found = [r[0] for r in conn.execute("SELECT title FROM tasks WHERE title LIKE ? ORDER BY id", (pattern,))]
check(7, found)

## 3. Transactions, GROUP BY, upsert

- `with conn:` is a transaction: it commits if the block succeeds and rolls back everything if it raises.
- Use it for changes that must happen together, like a transfer between two accounts.
- `SELECT done, COUNT(*) FROM tasks GROUP BY done` counts per group. `INSERT ... ON CONFLICT(key) DO UPDATE SET value = excluded.value` inserts or updates.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE accounts (name TEXT PRIMARY KEY, balance REAL CHECK (balance >= 0))")
conn.executemany("INSERT INTO accounts VALUES (?, ?)", [("a", 10), ("b", 0)])
conn.commit()
try:
    with conn:
        conn.execute("UPDATE accounts SET balance = balance + 50 WHERE name = 'b'")
        conn.execute("UPDATE accounts SET balance = balance - 50 WHERE name = 'a'")   # fails CHECK
except sqlite3.IntegrityError:
    print("rolled back")
print(conn.execute("SELECT * FROM accounts ORDER BY name").fetchall())

**Exercise 8.** After the failed transfer in the demo, what is `b`'s balance?

In [ ]:
predicted = ___
check(8, predicted)

**Exercise 9.** Count tasks per status with `GROUP BY`.

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE tasks (id INTEGER PRIMARY KEY, done INTEGER)")
conn.executemany("INSERT INTO tasks (done) VALUES (?)", [(0,), (1,), (0,), (0,)])
counts = dict(conn.execute("SELECT done, COUNT(*) FROM tasks GROUP BY done"))
predicted = ___
check(9, predicted)

**Exercise 10.** Upsert twice with the same key. How many rows, and which value wins?

In [ ]:
import sqlite3
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE settings (key TEXT PRIMARY KEY, value TEXT)")
sql = "INSERT INTO settings VALUES (?, ?) ON CONFLICT(key) DO UPDATE SET value = excluded.value"
conn.execute(sql, ("theme", "light"))
conn.execute(sql, ("theme", "dark"))
rows = conn.execute("SELECT * FROM settings").fetchall()
predicted = ___
check(10, predicted)

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### 1. `create_schema`

Create the `tasks` and `accounts` tables with their constraints.

In [ ]:
def create_schema(conn: sqlite3.Connection) -> None:
    """Create tables:
    tasks(id INTEGER PRIMARY KEY, title TEXT NOT NULL CHECK(trim(title) != ''),
          done INTEGER NOT NULL DEFAULT 0)
    accounts(name TEXT PRIMARY KEY, balance REAL NOT NULL CHECK(balance >= 0))
    """
    ___(
        """
        CREATE TABLE IF NOT EXISTS tasks (
            id INTEGER PRIMARY KEY,
            title TEXT NOT NULL CHECK (trim(title) != ''),
            done INTEGER NOT NULL DEFAULT 0
        );
        CREATE TABLE IF NOT EXISTS accounts (
            name TEXT PRIMARY KEY,
            balance REAL NOT NULL CHECK (balance >= 0)
        );
        """
    )

In [ ]:
check('create_schema')

### 2. `add_task`

Insert a task with a `?` placeholder and return its id.

In [ ]:
def add_task(conn: sqlite3.Connection, title: str) -> int:
    """Insert a task using a ? placeholder (never string formatting) and return its id."""
    with conn:
        cursor = ___
    return cursor.lastrowid

In [ ]:
check('add_task')

### 3. `list_tasks`

`(id, title, done)` tuples by id, optionally filtered by `done`.

In [ ]:
def list_tasks(conn: sqlite3.Connection, done=None) -> list[tuple[int, str, bool]]:
    """Return (id, title, done) tuples ordered by id; filter by done when it is not None."""
    sql, params = "SELECT id, title, done FROM tasks", ()
    if ___:
        sql, params = sql + " WHERE done = ?", (int(done),)
    return ___

In [ ]:
check('list_tasks')

### 4. `complete_task`

Mark a task done; `False` if it doesn't exist.

In [ ]:
def complete_task(conn: sqlite3.Connection, task_id: int) -> bool:
    """Mark a task done. Return False if no task has that id."""
    with conn:
        return ___

In [ ]:
check('complete_task')

### 5. `delete_task`

Delete a task; `False` if it doesn't exist.

In [ ]:
def delete_task(conn: sqlite3.Connection, task_id: int) -> bool:
    """Delete a task. Return False if no task has that id."""
    with conn:
        return ___

In [ ]:
check('delete_task')

### 6. `transfer`

Move money atomically; raise `ValueError` and change nothing on failure.

In [ ]:
def transfer(conn: sqlite3.Connection, source: str, target: str, amount: float) -> None:
    """Move money between accounts atomically.

    If the CHECK constraint fails (overdraft) or an account is missing, raise
    ValueError and leave BOTH balances unchanged. Hint: `with conn:` commits or rolls back.
    """
    try:
        with conn:
            for name, delta in ___:
                updated = conn.execute("UPDATE accounts SET balance = balance + ? WHERE name = ?",
                                       (delta, name)).rowcount
                if ___:
                    raise ValueError(f"unknown account: {name}")
    except sqlite3.IntegrityError as error:
        raise ValueError("insufficient funds") from error

In [ ]:
check('transfer')

### Independent build — `count_by_status`

Return {'open': n, 'done': m} for the tasks table. The done column contains 0 for open tasks and 1 for completed tasks. Include both keys even if one or both counts are zero.

Implement the interface below, run its check, and explain one boundary case with a test of your own.

In [ ]:
def count_by_status(conn: sqlite3.Connection) -> dict[str, int]:
    raise NotImplementedError

In [ ]:
def acceptance_cases(submission):
    import sqlite3
    from contextlib import closing
    with closing(sqlite3.connect(":memory:")) as db:
        db.execute("CREATE TABLE tasks(id INTEGER PRIMARY KEY, title TEXT, done INTEGER)")
        assert submission(db) == {"open": 0, "done": 0}, "Include both statuses for an empty table."
        db.executemany("INSERT INTO tasks (title, done) VALUES (?, ?)", [("pack", 1), ("pay", 1)])
        assert submission(db) == {"open": 0, "done": 2}, "Count completed tasks when none are open."
        db.execute("INSERT INTO tasks (title, done) VALUES ('ship', 0)")
        assert submission(db) == {"open": 1, "done": 2}, "Keep both status counts."

check.assess('count_by_status', acceptance_cases)

### 8. `search_tasks`

Titles containing the text, case-insensitive, by id.

In [ ]:
def search_tasks(conn: sqlite3.Connection, text: str) -> list[str]:
    """Titles containing text (case-insensitive), ordered by id. Use LIKE with a ? parameter."""
    rows = conn.execute("SELECT title FROM tasks WHERE title LIKE ? ORDER BY id", (f"%{text}%",))
    return ___

In [ ]:
check('search_tasks')

### 9. `set_setting`

Insert or update a setting (an upsert).

In [ ]:
def set_setting(conn: sqlite3.Connection, key: str, value: str) -> None:
    """Insert or update a row in settings(key TEXT PRIMARY KEY, value TEXT) — an 'upsert'.

    Hint: INSERT ... ON CONFLICT(key) DO UPDATE SET value = excluded.value
    """
    with conn:
        conn.execute("CREATE TABLE IF NOT EXISTS settings (key TEXT PRIMARY KEY, value TEXT)")
        conn.execute("INSERT INTO settings (key, value) VALUES (?, ?) "
                     "ON CONFLICT(key) DO UPDATE SET value = excluded.value", ___)

In [ ]:
check('set_setting')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** Why write `execute('… WHERE id = ?', (task_id,))`?

   - **a)** Faster typing
   - **b)** SQLite requires it
   - **c)** Prevents SQL injection

**2.** `with conn:` in sqlite3…

   - **a)** Closes the connection
   - **b)** Opens a new database
   - **c)** Commits on success, rolls back on error

**3.** A CHECK constraint is enforced by…

   - **a)** Nobody
   - **b)** Your Python code
   - **c)** The database itself

In [ ]:
from quiz import Quiz

quiz = Quiz('intermediate/15', {1: '95bcf96ede2a', 2: '7a3655181fa2', 3: '4531163d339b'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [16 — HTTP, JSON APIs, and Reliable Clients](16_http_apis_and_clients.ipynb)